# 阶段 4 · A：诊断论文原版 1.5B 的服务器 1 塌缩（Colab）

B（从我们的硬标签 1.5B 出发）在另一个 notebook `S4_B_硬标签起点.ipynb`。

起点是原作者的 **sft 1.5B**（论文格式：长提示词 + 中文动作名）。一共四次运行，每次只改第 3 格的 `RUN`，然后「代码执行程序 → 全部运行」：

| RUN | 做什么 | G4 上预计（估算） |
|---|---|---|
| `diagnose` | 不训练。原作者 sft / grpo / score-grpo 三个模型，算出每条状态下**每个动作的完整概率**，再和贪心解码的答案对比，对照论文表 V | 约 15–25 分钟 |
| `smoke` | GRPO 冒烟：fixed 奖励，30 步 | 约 10–15 分钟 |
| `fixed` | GRPO 正式：fixed 奖励（非法回答比最差的合法动作还差 5 槽），2 轮 | 约 50–70 分钟 |
| `repo` | GRPO 正式：repo 奖励（原作者仓库的写法：非法回答只记 5.0 槽，认出动作 +1），其余设置和 fixed 完全一样 | 约 50–70 分钟 |

预计时长的依据：阶段 3 论文原版基线在 G4 上实测每个模型 3.1 分钟答完 5663 条（同样长度的提示词）；GRPO 每步要采样 64 个回答再前向、反向，按它推算。

跑之前：运行时选 **G4**（没有就 L4 或 A100）。Drive 里要有：
- `MyDrive/COMLLM/s3_data/`（阶段 3 的测试集，不用动）
- `MyDrive/COMLLM/s4_data/`（Mac 上 `python -m train.make_grpo_data` 生成的整个文件夹）
- `MyDrive/COMLLM/reference_model/` 下原作者的三个模型

**结果**在 Drive 的 `MyDrive/COMLLM/s4_runs/<RUN>_<UTC 时间>/`。

**止损线**：
- 训练第 5 步会打印预计时长。超过 1.5 小时就停下来告诉我。
- `smoke` 跑完打印「冒烟判据」。三条里有一条不过，先别跑 `fixed`，把第 8 格的输出截图发到对话里。最晚 11/11 冒烟还不过，阶段 4 停。
- 任何一格报错就停，截图发到对话里。
- 跑完「代码执行程序 → 断开连接并删除运行时」。

In [ ]:
# 第 1 格：安装固定版本的库（和阶段 3 一样）
import subprocess, sys
try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "transformers==5.20.2", "peft==0.21.2", "accelerate", "sentencepiece"], check=True)
    # Colab 预装的 torchao 版本太老，peft 0.21 检测到它会直接报错。这里用不到 torchao，卸掉
    subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=False)

import importlib, importlib.util
importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao 还在：菜单「代码执行程序 → 重新启动会话」后再「全部运行」"
print("依赖安装完成")

In [ ]:
# 第 2 格：检查 GPU。G4 / L4 / A100 支持 bf16；T4 不支持，这个 notebook 不在 T4 上跑
import os, time, json, math, hashlib, csv
from datetime import datetime, timezone
import numpy as np
import torch

if IN_COLAB:
    subprocess.run(["nvidia-smi"], check=False)
    assert torch.cuda.is_available(), "没检测到 GPU：菜单「代码执行程序 → 更改运行时类型」选 G4 后再运行"
HAS_GPU = torch.cuda.is_available()
DEVICE = "cuda" if HAS_GPU else "cpu"
if HAS_GPU:
    assert torch.cuda.is_bf16_supported(), "这块 GPU 不支持 bf16（T4 就不支持）：换成 G4 或 L4"
DTYPE = torch.bfloat16 if HAS_GPU else torch.float32
GPU_NAME = torch.cuda.get_device_name() if HAS_GPU else "无（CPU）"
print("GPU:", GPU_NAME, "| 精度:", "bf16" if HAS_GPU else "fp32", "| torch", torch.__version__)

In [ ]:
# 第 3 格：这次跑哪个。每次只改下面一行
RUN = "diagnose"       # 依次：diagnose → smoke → fixed → repo

# ---- 下面不用改 ----
PRESETS = {
    "diagnose": {"train": False},
    "smoke": {"train": True, "reward": "fixed", "max_steps": 30, "eval_every": 10, "final_eval": False},
    "fixed": {"train": True, "reward": "fixed", "epochs": 2, "eval_every": 25, "final_eval": True},
    "repo": {"train": True, "reward": "repo", "epochs": 2, "eval_every": 25, "final_eval": True},
}
REF_MODELS = ["sft", "grpo", "score-grpo"]      # diagnose 看这三个
START = "sft"                                   # GRPO 从哪个原作者模型出发
REF_NAME = "qwen2.5-1.5b-mec-{}-final-all"      # Drive 上模型文件夹的名字

# GRPO 超参数：smoke / fixed / repo 完全一样，只有奖励不同
G = 8                      # 每道题采样几个回答（论文式 17 的 G）
PROMPTS_PER_STEP = 8       # 每步几道题：一步一共 8 × 8 = 64 个回答
TEMP = 1.3                 # 采样温度
TOP_P = 0.95               # 采样时只在累计概率前 95% 的 token 里抽
BETA = 0.04                # KL 惩罚的系数 β（论文式 19）
LR = 5e-5                  # 学习率（前 WARMUP_STEPS 步线性升上去，之后不变）
WARMUP_STEPS = 10
EPS_CLIP = 0.2             # 论文式 19 的 clip 范围 ε
LORA_R, LORA_ALPHA = 16, 32
MAX_NEW_TOKENS = 16        # "卸载到边缘服务器11" 约 8 个 token，留余量
LOSS_CHUNK = 16            # 算损失时每次前向多少个回答（显存不够就改小）
GEN_BATCH = 32             # 贪心评测时每批几条
SCORE_ROWS = 2             # 算完整动作概率时每批几条状态（每条状态要算 台数+1 个候选）
N_PER_COUNT = 200          # 最终评测：test_seen 每种台数取前 200 条
N_DIST_11 = 200            # 动作分布：test_seen 里 11 台的前 200 条（对照论文表 V）
DIST_TEMPS = (1.0, 0.7)    # 算动作概率用的温度：1.0 = 模型本身；0.7 = 原作者评测脚本的采样温度
SEED = 0

# 本机测试用（Colab 上这些环境变量都不存在，不起作用）
LOCAL_MODEL = os.environ.get("S4_LOCAL_MODEL")
if os.environ.get("S4_RUN"):
    RUN = os.environ["S4_RUN"]
P = dict(PRESETS[RUN])
if os.environ.get("S4_MAX_STEPS"):
    P["max_steps"] = int(os.environ["S4_MAX_STEPS"])
for name in ("N_PER_COUNT", "N_DIST_11", "PROMPTS_PER_STEP", "G", "LOSS_CHUNK"):
    if os.environ.get("S4_" + name):
        globals()[name] = int(os.environ["S4_" + name])

STAMP = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DATA3 = "/content/drive/MyDrive/COMLLM/s3_data"
    DATA4 = "/content/drive/MyDrive/COMLLM/s4_data"
    REF_DIR = "/content/drive/MyDrive/COMLLM/reference_model"
    OUT = f"/content/drive/MyDrive/COMLLM/s4_runs/{RUN}_{STAMP}"
else:
    DATA3 = os.environ["S4_DATA3"]
    DATA4 = os.environ["S4_DATA4"]
    REF_DIR = None
    OUT = os.path.join(os.environ["S4_OUT"], f"{RUN}_{STAMP}")
os.makedirs(OUT, exist_ok=True)


def model_dir(m):
    if LOCAL_MODEL:
        return LOCAL_MODEL
    return os.path.join(REF_DIR, REF_NAME.format(m))


for m in REF_MODELS:
    assert os.path.exists(os.path.join(model_dir(m), "config.json")), f"找不到模型文件夹 {model_dir(m)}"
print("这次跑:", RUN, P)
print("结果目录:", OUT)

In [ ]:
# 第 4 格：读数据，核对校验值；读奖励函数（和 Mac 仓库里 train/grpo_reward.py 是同一个文件）
def read_jsonl(path):
    rows = []
    with open(path, encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows


def md5_of(path):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def load_checked(data_dir, names):
    """读 meta.json，逐个核对 jsonl 的校验值，再读进来；每行记下它在文件里是第几行（_idx）。"""
    with open(os.path.join(data_dir, "meta.json"), encoding="utf-8") as f:
        meta = json.load(f)
    out = {}
    for name in names:
        path = os.path.join(data_dir, name + ".jsonl")
        assert md5_of(path) == meta["files"][name]["md5"], f"{name}.jsonl 的校验值和 meta.json 对不上：重新上传整个文件夹"
        out[name] = read_jsonl(path)
        for i, r in enumerate(out[name]):
            r["_idx"] = i
            r["split"] = name
        print(f"{name:13s} {len(out[name]):5d} 条 ✓")
    return meta, out


META3, D3 = load_checked(DATA3, ["friend_clean", "test_seen"])
META4, D4 = load_checked(DATA4, ["grpo_train", "grpo_val"])
DATA = {}
for d in (D3, D4):
    for name in d:
        DATA[name] = d[name]

assert md5_of(os.path.join(DATA4, "grpo_reward.py")) == META4["reward_md5"], "grpo_reward.py 和 meta.json 对不上：重新上传 s4_data"
sys.path.insert(0, DATA4)
import grpo_reward as R
importlib.reload(R)

INSTR = META3["paper_instruction"]
REPO_SYSTEM = "你是一个MEC任务卸载专家。请严格按照用户的指示分析输入数据，输出卸载策略。不要包含任何解释、代码块标记或额外文本。"
INPUT_MARKER = "**输入数据**："


def chatml(messages):
    """和仓库里 deploy/prompts.py 的 chatml 一模一样。"""
    text = ""
    for m in messages:
        text = text + "<|im_start|>" + m["role"] + "\n" + m["content"] + "<|im_end|>\n"
    text = text + "<|im_start|>assistant\n"
    return text


def mec_lines(inp):
    """状态字典 → 每行一个 "键: 值"；值是列表或字典时写成 JSON（中文原样）。"""
    lines = []
    for key in inp:
        value = inp[key]
        if isinstance(value, (list, dict)):
            value = json.dumps(value, ensure_ascii=False)
        lines.append(key + ": " + str(value))
    return "\n".join(lines)


def repo_messages(row):
    """原作者模型训练时用的对话：系统提示 + 说明 + 空一行 + "**输入数据**：" + 每行一个 "键: 值"。"""
    return [{"role": "system", "content": REPO_SYSTEM},
            {"role": "user", "content": INSTR + "\n\n" + INPUT_MARKER + "\n" + mec_lines(row["paper_input"])}]


def candidates(n_servers):
    """n 台服务器时所有合法动作的文字，下标就是动作编号。"""
    out = ["本地执行"]
    for k in range(1, n_servers + 1):
        out.append("卸载到边缘服务器" + str(k))
    return out


# 最终评测用的行：原作者干净测试集全部 + test_seen 每种台数前 N_PER_COUNT 条
EVAL_ROWS = list(DATA["friend_clean"])
count = {}
for r in DATA["test_seen"]:
    count[r["servers"]] = count.get(r["servers"], 0) + 1
    if count[r["servers"]] <= N_PER_COUNT:
        EVAL_ROWS.append(r)
# 算动作分布的行：原作者干净测试集全部 + test_seen 里 11 台的前 N_DIST_11 条
DIST_ROWS = list(DATA["friend_clean"])
n11 = 0
for r in DATA["test_seen"]:
    if r["servers"] == 11 and n11 < N_DIST_11:
        DIST_ROWS.append(r)
        n11 += 1
print(f"最终评测 {len(EVAL_ROWS)} 条，动作分布 {len(DIST_ROWS)} 条（其中 test_seen 11 台 {n11} 条）")
for k, v in META4["labels"]["grpo_train"].items():
    print(f"训练题 {k} 台：{v['n']} 条")

In [ ]:
# 第 5 格：读模型、拼提示词、贪心作答、算每个动作的完整概率（这一格只定义函数，不跑东西）
from transformers import AutoModelForCausalLM, AutoTokenizer, GenerationConfig

TOK = None          # 三个原作者模型的分词器完全一样，读一次就够
EVAL_DEADLINE = 10.0    # 开环评测的截止期（槽），和 eval/open_loop.py 一样


def load_tokenizer():
    """读分词器，并核对：模型自带的对话模板拼出来的 = chatml 拼的；提示词和动作分开切 token = 连起来切。"""
    tk = AutoTokenizer.from_pretrained(model_dir(START))
    tk.padding_side = "left"
    if tk.pad_token is None:
        tk.pad_token = "<|endoftext|>"
    r = DATA["friend_clean"][0]
    hf = tk.apply_chat_template(repo_messages(r), tokenize=False, add_generation_prompt=True)
    assert hf == chatml(repo_messages(r)), "模型自带的对话模板和 chatml 拼的不一样"
    for r in DATA["friend_clean"][:20]:
        p = chatml(repo_messages(r))
        for c in candidates(r["servers"]):
            joint = tk(p + c, add_special_tokens=False)["input_ids"]
            apart = tk(p, add_special_tokens=False)["input_ids"] + tk(c, add_special_tokens=False)["input_ids"]
            assert joint == apart, "提示词和动作名连起来切 token 时粘在了一起"
    return tk


def encode_prompts(rows):
    """每行的提示词全文切成 token，存进 r["p_ids"]（已经有的跳过）。"""
    todo = []
    for r in rows:
        if "p_ids" not in r:
            todo.append(r)
    texts = []
    for r in todo:
        texts.append(chatml(repo_messages(r)))
    if texts:
        ids = TOK(texts, add_special_tokens=False)["input_ids"]
        for r, x in zip(todo, ids):
            r["p_ids"] = x


def load_ref(m):
    """读一个原作者模型。grpo 两个存的是 fp16，读进来统一转成 bf16。"""
    t0 = time.time()
    md = AutoModelForCausalLM.from_pretrained(model_dir(m), dtype=DTYPE).to(DEVICE)
    md.eval()
    print(f"读入 {m}：{time.time() - t0:.0f} 秒")
    return md


def batches_by_count(rows, size, rng=None):
    """按服务器台数分组（同组提示词长度几乎一样），每组切成 size 条一批；给了 rng 就打乱。"""
    groups = {}
    for r in rows:
        groups.setdefault(r["servers"], []).append(r)
    out = []
    for k in sorted(groups):
        g = groups[k]
        if rng is not None:
            order = rng.permutation(len(g))
            g = [g[int(i)] for i in order]
        for s in range(0, len(g), size):
            out.append(g[s:s + size])
    if rng is not None:
        order = rng.permutation(len(out))
        out = [out[int(i)] for i in order]
    return out


def left_pad(id_lists):
    """左补齐：返回 (ids, mask)，形状都是 (条数, 最长长度)。"""
    width = 0
    for x in id_lists:
        if len(x) > width:
            width = len(x)
    ids = torch.full((len(id_lists), width), TOK.pad_token_id, dtype=torch.long)
    mask = torch.zeros((len(id_lists), width), dtype=torch.long)
    for i, x in enumerate(id_lists):
        ids[i, width - len(x):] = torch.tensor(x, dtype=torch.long)
        mask[i, width - len(x):] = 1
    return ids.to(DEVICE), mask.to(DEVICE)


def stop_ids():
    return [TOK.convert_tokens_to_ids("<|im_end|>"), TOK.eos_token_id]


@torch.no_grad()
def greedy_answers(md, rows):
    """贪心解码（每步取概率最大的 token），返回 {(split, 行号): (动作, 原文)}；动作 −1 = 非法。"""
    md.eval()
    encode_prompts(rows)
    cfg = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=False,
                           pad_token_id=TOK.pad_token_id, eos_token_id=stop_ids())
    out = {}
    for batch in batches_by_count(rows, GEN_BATCH):
        ids, mask = left_pad([r["p_ids"] for r in batch])
        gen = md.generate(input_ids=ids, attention_mask=mask, generation_config=cfg)
        new = gen[:, ids.shape[1]:]
        for i, r in enumerate(batch):
            text = TOK.decode(new[i], skip_special_tokens=True).strip()
            kind, a = R.parse(text, r["servers"])
            out[(r["split"], r["_idx"])] = (a, text)
    return out


@torch.no_grad()
def action_logprobs(md, rows):
    """每条状态下，模型说出每个合法动作（整句，含结尾的 <|im_end|>）的 log 概率。

    返回 {(split, 行号): {温度: [每个动作的 log 概率]}}。
    1. 每条状态 × 每个候选动作，拼成"提示词 + 动作名 + <|im_end|>"
    2. 提示词部分左补齐、动作部分右补齐，一批前向
    3. 只取动作那几个位置的 logits，除以温度后取 log_softmax，再取实际那个 token 的值
    4. 一个候选的所有 token 加起来 = 整句的 log 概率
    """
    md.eval()
    encode_prompts(rows)
    im_end = TOK.convert_tokens_to_ids("<|im_end|>")
    cand_cache = {}
    out = {}
    for batch in batches_by_count(rows, SCORE_ROWS):
        prompts = []
        conts = []
        owners = []
        for r in batch:
            n = r["servers"]
            if n not in cand_cache:
                cand_cache[n] = []
                for c in candidates(n):
                    cand_cache[n].append(TOK(c, add_special_tokens=False)["input_ids"] + [im_end])
            for a, c_ids in enumerate(cand_cache[n]):
                prompts.append(r["p_ids"])
                conts.append(c_ids)
                owners.append((r, a))
        p_ids, p_mask = left_pad(prompts)
        width = 0
        for c in conts:
            if len(c) > width:
                width = len(c)
        c_ids = torch.full((len(conts), width), TOK.pad_token_id, dtype=torch.long, device=DEVICE)
        c_mask = torch.zeros((len(conts), width), dtype=torch.long, device=DEVICE)
        for i, c in enumerate(conts):
            c_ids[i, :len(c)] = torch.tensor(c, dtype=torch.long)
            c_mask[i, :len(c)] = 1
        logits = completion_logits(md, p_ids, p_mask, c_ids, c_mask)
        for r in batch:
            out[(r["split"], r["_idx"])] = {}
            for T in DIST_TEMPS:
                out[(r["split"], r["_idx"])][T] = [0.0] * (r["servers"] + 1)
        for T in DIST_TEMPS:
            lp = torch.log_softmax(logits / T, dim=-1)
            tok_lp = lp.gather(-1, c_ids.unsqueeze(-1)).squeeze(-1)
            seq_lp = (tok_lp * c_mask).sum(dim=1).cpu().numpy()
            for i, (r, a) in enumerate(owners):
                out[(r["split"], r["_idx"])][T][a] = float(seq_lp[i])
    return out


def completion_logits(md, p_ids, p_mask, c_ids, c_mask):
    """一次前向，只返回回答那几个位置的 logits，形状 (条数, 回答长度, 词表)。

    位置 t 的 logits 预测的是第 t+1 个 token，所以要从提示词最后一个位置开始取，取到回答倒数第二个位置。
    logits_to_keep 让模型只在最后 (回答长度 + 1) 个位置算词表分数，长提示词也不会爆显存。
    """
    ids = torch.cat([p_ids, c_ids], dim=1)
    mask = torch.cat([p_mask, c_mask], dim=1)
    pos = (mask.cumsum(dim=1) - 1).clamp(min=0)
    width = c_ids.shape[1]
    with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=HAS_GPU):
        out = md(input_ids=ids, attention_mask=mask, position_ids=pos, logits_to_keep=width + 1)
    return out.logits[:, :-1, :].float()


def quick_metrics(rows, answers):
    """准确率、PR（论文式 28–29）、非法率、选服务器 1 的比例、选本地的比例。正式指标在 Mac 上用 eval.open_loop 算。"""
    hit = 0
    invalid = 0
    s1 = 0
    local = 0
    al = 0.0
    al_best = 0.0
    for r in rows:
        a = answers[(r["split"], r["_idx"])][0]
        if a < 0:
            invalid += 1
            al += EVAL_DEADLINE + R.RHO
        else:
            al += R.effective_cost(r["costs"][a], EVAL_DEADLINE)
            if a == r["best"]:
                hit += 1
            if a == 1:
                s1 += 1
            if a == 0:
                local += 1
        al_best += R.effective_cost(r["costs"][r["best"]], EVAL_DEADLINE)
    n = len(rows)
    return {"n": n, "acc_pct": round(100.0 * hit / n, 1), "pr_pct": round(100.0 * al_best / al, 1),
            "invalid_pct": round(100.0 * invalid / n, 1), "s1_pct": round(100.0 * s1 / n, 1),
            "local_pct": round(100.0 * local / n, 1)}


def dist_summary(rows, lps, greedy):
    """完整动作概率的汇总。

    1. 逐条：T=1 下整句概率最大的动作（"整句最大概率"）；它和贪心解码选的是不是同一个
    2. 合法动作的概率加起来是多少（剩下的就是说出别的话的概率）
    3. 采样温度 0.7 时，抽到最优动作的概率（= 原作者评测方式下的期望准确率）
    4. T=1 下服务器 1 的平均概率；11 台时还看服务器 1、10、11 合起来（它们的编号都以"1"开头）
    5. 11 台的 test_seen 行：温度 0.7 采样时每个动作的期望占比（对照论文表 V），以及贪心的占比
    """
    seq_hit = 0
    greedy_hit = 0
    agree = 0
    mass = 0.0
    exp_hit_07 = 0.0
    p_s1 = 0.0
    p_prefix1_11 = 0.0
    n11_all = 0
    share07 = [0.0] * 12
    share_greedy = [0] * 12
    n11 = 0
    for r in rows:
        key = (r["split"], r["_idx"])
        lp1 = lps[key][1.0]
        lp07 = lps[key][0.7]
        best_seq = 0
        for a in range(len(lp1)):
            if lp1[a] > lp1[best_seq]:
                best_seq = a
        g = greedy[key][0]
        if best_seq == r["best"]:
            seq_hit += 1
        if g == r["best"]:
            greedy_hit += 1
        if g == best_seq:
            agree += 1
        for a in range(len(lp1)):
            mass += math.exp(lp1[a])
        exp_hit_07 += math.exp(lp07[r["best"]])
        p_s1 += math.exp(lp1[1])
        if r["servers"] == 11:
            n11_all += 1
            p_prefix1_11 += math.exp(lp1[1]) + math.exp(lp1[10]) + math.exp(lp1[11])
            if r["split"] == "test_seen":
                n11 += 1
                total = 0.0
                for a in range(12):
                    total += math.exp(lp07[a])
                for a in range(12):
                    share07[a] += math.exp(lp07[a]) / total
                if g >= 0:
                    share_greedy[g] += 1
    n = len(rows)
    s = {"n": n,
         "seq_argmax_acc_pct": round(100.0 * seq_hit / n, 1),
         "greedy_acc_pct": round(100.0 * greedy_hit / n, 1),
         "greedy_equals_seq_argmax_pct": round(100.0 * agree / n, 1),
         "legal_mass_pct": round(100.0 * mass / n, 1),
         "expected_acc_T0.7_pct": round(100.0 * exp_hit_07 / n, 1),
         "mean_p_server1_pct": round(100.0 * p_s1 / n, 1)}
    if n11_all:
        s["mean_p_server1_10_11_pct_at_11"] = round(100.0 * p_prefix1_11 / n11_all, 1)
    if n11:
        s["table5_T0.7_share_pct"] = [round(100.0 * x / n11, 1) for x in share07]
        s["table5_greedy_share_pct"] = [round(100.0 * x / n11, 1) for x in share_greedy]
        s["table5_n"] = n11
    return s


def write_dist(path, rows, lps, greedy):
    """每条状态一行：动作概率（两个温度）、贪心答案、最优动作。"""
    with open(path, "w", encoding="utf-8") as f:
        for r in rows:
            key = (r["split"], r["_idx"])
            rec = {"split": r["split"], "idx": r["_idx"], "servers": r["servers"], "best": r["best"],
                   "greedy": greedy[key][0], "greedy_raw": greedy[key][1]}
            for T in DIST_TEMPS:
                rec["logp_T" + str(T)] = [round(x, 4) for x in lps[key][T]]
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")


def rows_of(rows, split):
    """rows 里属于某个集合的那些行。"""
    out = []
    for r in rows:
        if r["split"] == split:
            out.append(r)
    return out


def trainable(md):
    """模型里要训练的参数（只有 LoRA 的小矩阵）。"""
    out = []
    for p in md.parameters():
        if p.requires_grad:
            out.append(p)
    return out


def write_preds(path, rows, answers):
    """Mac 上 eval.open_loop 读的格式：split, idx, pred（−1 = 非法）, raw。"""
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.writer(f)
        w.writerow(["split", "idx", "pred", "raw"])
        for r in rows:
            a, text = answers[(r["split"], r["_idx"])]
            w.writerow([r["split"], r["_idx"], a, text])


def seq_argmax_answers(rows, lps):
    """用"整句最大概率"的动作当答案，格式和 greedy_answers 一样。"""
    out = {}
    for r in rows:
        lp1 = lps[(r["split"], r["_idx"])][1.0]
        best = 0
        for a in range(len(lp1)):
            if lp1[a] > lp1[best]:
                best = a
        out[(r["split"], r["_idx"])] = (best, candidates(r["servers"])[best])
    return out

In [ ]:
# 第 6 格：GRPO 的一步（这一格只定义函数，不跑东西）
from peft import LoraConfig, get_peft_model


@torch.no_grad()
def sample_groups(md, batch):
    """每道题采样 G 个回答（论文式 17）。

    1. 提示词左补齐，送进 generate：开采样，温度 TEMP，只在前 TOP_P 的 token 里抽，每道题 G 个
       返回的顺序是：题 1 的 G 个回答、题 2 的 G 个回答……
    2. 回答部分：到第一个 <|im_end|>（含）为止算有效，后面的位置标成无效
    3. 有效部分解码成文字
    """
    md.eval()
    p_ids, p_mask = left_pad([r["p_ids"] for r in batch])
    cfg = GenerationConfig(max_new_tokens=MAX_NEW_TOKENS, do_sample=True, temperature=TEMP, top_p=TOP_P,
                           num_return_sequences=G, pad_token_id=TOK.pad_token_id, eos_token_id=stop_ids())
    gen = md.generate(input_ids=p_ids, attention_mask=p_mask, generation_config=cfg)
    c_ids = gen[:, p_ids.shape[1]:]
    stops = stop_ids()
    c_mask = torch.zeros_like(c_ids)
    texts = []
    for i in range(c_ids.shape[0]):
        n = 0
        for t in range(c_ids.shape[1]):
            n += 1
            if int(c_ids[i, t]) in stops:
                break
        c_mask[i, :n] = 1
        texts.append(TOK.decode(c_ids[i, :n], skip_special_tokens=True).strip())
    p_ids = p_ids.repeat_interleave(G, dim=0)
    p_mask = p_mask.repeat_interleave(G, dim=0)
    return p_ids, p_mask, c_ids, c_mask, texts


def token_logps(md, p_ids, p_mask, c_ids, c_mask):
    """回答里每个 token 的 log 概率，形状 (回答数, 回答长度)。和采样时一样先除以温度 TEMP。"""
    logits = completion_logits(md, p_ids, p_mask, c_ids, c_mask)
    lp = torch.log_softmax(logits / TEMP, dim=-1)
    return lp.gather(-1, c_ids.unsqueeze(-1)).squeeze(-1)


def grpo_step(md, opt, batch, style, step):
    """一步 GRPO，返回这一步的统计。

    1. 每道题采样 G 个回答
    2. 每个回答算奖励（style = fixed 或 repo），解析出动作
    3. 组内归一化得到优势 A（论文式 18）；奖励全相同的组优势全是 0
    4. 分块前向，算论文式 19 的损失：
         比值 ρ = π_θ / π_θold。这里每批回答只更新一次，更新前 π_θ 就是 π_θold，所以 ρ = 1，
         clip 不起作用，损失的梯度 = −A × ∇log π_θ（优势为正的回答概率往上推，为负的往下压）
         KL 用 exp(ref − lp) − (ref − lp) − 1 逐 token 估计，ref = 关掉 LoRA 的模型 = 原作者 sft
         每个回答内对 token 取平均，再对全部 G × 题数 个回答取平均
    5. 梯度裁剪到 1.0 → 设学习率 → AdamW 更新
    """
    # 1. 采样
    p_ids, p_mask, c_ids, c_mask, texts = sample_groups(md, batch)

    # 2. 奖励和动作
    rewards = []
    kinds = []
    actions = []
    for i, text in enumerate(texts):
        r = batch[i // G]
        rewards.append(R.reward(style, text, r["costs"], r["state"]["deadline"]))
        kind, a = R.parse(text, r["servers"])
        kinds.append(kind)
        actions.append(a)

    # 3. 优势
    adv, zero_std = R.group_advantages(rewards, G)
    adv_t = torch.tensor(adv, dtype=torch.float32, device=DEVICE)

    # 4. 损失和梯度
    #    4a. 先算参考模型（关掉 LoRA = 原作者 sft）在每个回答上的 log 概率，不需要梯度
    n_all = len(texts)
    refs = []
    if BETA > 0:
        md.eval()
        with torch.no_grad():
            with md.disable_adapter():
                for s in range(0, n_all, LOSS_CHUNK):
                    e = min(n_all, s + LOSS_CHUNK)
                    refs.append(token_logps(md, p_ids[s:e], p_mask[s:e], c_ids[s:e], c_mask[s:e]))
    #    4b. 再分块算当前模型的 log 概率和损失，反向
    md.train()
    loss_sum = 0.0
    kl_sum = 0.0
    chunk = 0
    for s in range(0, n_all, LOSS_CHUNK):
        e = min(n_all, s + LOSS_CHUNK)
        cm = c_mask[s:e].float()
        lp = token_logps(md, p_ids[s:e], p_mask[s:e], c_ids[s:e], c_mask[s:e])
        ratio = torch.exp(lp - lp.detach())
        a = adv_t[s:e].unsqueeze(1)
        per_tok = -torch.minimum(ratio * a, torch.clamp(ratio, 1.0 - EPS_CLIP, 1.0 + EPS_CLIP) * a)
        kl = torch.zeros_like(lp)
        if BETA > 0:
            ref = refs[chunk]
            kl = torch.exp(ref - lp) - (ref - lp) - 1.0
            per_tok = per_tok + BETA * kl
        n_tok = cm.sum(dim=1).clamp(min=1.0)
        per_seq = (per_tok * cm).sum(dim=1) / n_tok
        loss = per_seq.sum() / n_all
        loss.backward()
        loss_sum += float(loss.item())
        kl_sum += float(((kl.detach() * cm).sum(dim=1) / n_tok).sum().item())
        chunk += 1

    # 5. 更新
    torch.nn.utils.clip_grad_norm_(trainable(md), 1.0)
    lr = LR * min(1.0, (step + 1) / WARMUP_STEPS)
    for g in opt.param_groups:
        g["lr"] = lr
    opt.step()
    opt.zero_grad(set_to_none=True)

    # 统计
    n_ok = 0
    n_bad = 0
    n_none = 0
    n_s1 = 0
    n_local = 0
    top_share = 0.0
    for i in range(n_all):
        if kinds[i] == R.OK:
            n_ok += 1
            if actions[i] == 1:
                n_s1 += 1
            if actions[i] == 0:
                n_local += 1
        elif kinds[i] == R.BAD_SERVER:
            n_bad += 1
        else:
            n_none += 1
    for g0 in range(0, n_all, G):
        cnt = {}
        for i in range(g0, g0 + G):
            cnt[actions[i]] = cnt.get(actions[i], 0) + 1
        top = 0
        for v in cnt.values():
            if v > top:
                top = v
        top_share += top / G
    hit = 0
    for i in range(n_all):
        if actions[i] == batch[i // G]["best"]:
            hit += 1
    return {"step": step + 1, "loss": round(loss_sum, 5), "reward_mean": round(float(np.mean(rewards)), 4),
            "sample_acc_pct": round(100.0 * hit / n_all, 1),
            "zero_std_frac": round(zero_std / len(batch), 3), "kl": round(kl_sum / n_all, 5),
            "bad_server_pct": round(100.0 * n_bad / n_all, 1), "unparsed_pct": round(100.0 * n_none / n_all, 1),
            "s1_pct": round(100.0 * n_s1 / n_all, 1), "local_pct": round(100.0 * n_local / n_all, 1),
            "group_top_share": round(top_share / len(batch), 3), "lr": lr}

In [ ]:
# 第 7 格：开始跑
torch.manual_seed(SEED)
np.random.seed(SEED)
TOK = load_tokenizer()
SUMMARY = {"run": RUN, "gpu": GPU_NAME, "torch": torch.__version__, "settings": {}}

if not P["train"]:
    # ===== diagnose：三个原作者模型，贪心答案 + 完整动作概率 =====
    for m in REF_MODELS:
        print(f"===== {m} =====")
        t0 = time.time()
        md = load_ref(m)
        greedy = greedy_answers(md, DIST_ROWS)
        lps = action_logprobs(md, DIST_ROWS)
        run_dir = os.path.join(OUT, m)
        os.makedirs(run_dir, exist_ok=True)
        write_dist(os.path.join(run_dir, "dist.jsonl"), DIST_ROWS, lps, greedy)
        write_preds(os.path.join(run_dir, "preds.csv"), DIST_ROWS, seq_argmax_answers(DIST_ROWS, lps))
        with open(os.path.join(run_dir, "config.json"), "w", encoding="utf-8") as f:
            json.dump({"name": "论文原版 " + m + "（整句最大概率）", "model": REF_NAME.format(m),
                       "model_tag": "论文原版 1.5B", "decode": "seq_argmax", "n_rows": len(DIST_ROWS)},
                      f, ensure_ascii=False, indent=2)
        s = {}
        for split in ("friend_clean", "test_seen"):
            s[split] = dist_summary(rows_of(DIST_ROWS, split), lps, greedy)
        SUMMARY[m] = s
        print(json.dumps(s, ensure_ascii=False, indent=1))
        print(f"{m} 完成，用时 {(time.time() - t0) / 60:.1f} 分钟")
        del md
        if HAS_GPU:
            torch.cuda.empty_cache()
else:
    # ===== smoke / fixed / repo：从原作者 sft 出发做 GRPO =====
    style = P["reward"]
    SUMMARY["settings"] = {"start": START, "reward": style, "G": G, "prompts_per_step": PROMPTS_PER_STEP,
                           "temp": TEMP, "top_p": TOP_P, "beta": BETA, "lr": LR, "warmup": WARMUP_STEPS,
                           "lora_r": LORA_R, "lora_alpha": LORA_ALPHA, "max_new_tokens": MAX_NEW_TOKENS}
    md = load_ref(START)
    cfg = LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.0, target_modules="all-linear",
                     task_type="CAUSAL_LM")
    md = get_peft_model(md, cfg)
    md.print_trainable_parameters()
    md.gradient_checkpointing_enable()
    md.enable_input_require_grads()
    opt = torch.optim.AdamW(trainable(md), lr=LR, weight_decay=0.0)

    train_rows = DATA["grpo_train"]
    val_rows = DATA["grpo_val"]
    encode_prompts(train_rows)
    encode_prompts(val_rows)
    rng = np.random.default_rng(SEED)
    batches = []
    for ep in range(P.get("epochs", 1)):
        for b in batches_by_count(train_rows, PROMPTS_PER_STEP, rng):
            if len(b) == PROMPTS_PER_STEP:
                batches.append(b)
    total = len(batches)
    if P.get("max_steps"):
        total = min(total, P["max_steps"])

    log_f = open(os.path.join(OUT, "train_log.csv"), "w", newline="", encoding="utf-8")
    log_w = None
    VAL_LOG = []

    def run_val(step):
        v = quick_metrics(val_rows, greedy_answers(md, val_rows))
        v["step"] = step
        VAL_LOG.append(v)
        print(f"  验证集（贪心，{v['n']} 条）第 {step} 步：准确率 {v['acc_pct']}%  PR {v['pr_pct']}%  "
              f"非法 {v['invalid_pct']}%  服务器 1 {v['s1_pct']}%  本地 {v['local_pct']}%")

    run_val(0)
    t0 = time.time()
    STEP_LOG = []
    for step in range(total):
        st = grpo_step(md, opt, batches[step], style, step)
        st["elapsed_s"] = round(time.time() - t0, 1)
        STEP_LOG.append(st)
        if log_w is None:
            log_w = csv.DictWriter(log_f, fieldnames=list(st.keys()))
            log_w.writeheader()
        log_w.writerow(st)
        log_f.flush()
        if step == 4:
            eta = (time.time() - t0) / 5 * total
            print(f"前 5 步用了 {time.time() - t0:.0f} 秒，预计训练共约 {eta / 60:.0f} 分钟（估算）"
                  + ("  ← 超过 1.5 小时，停下来告诉我" if eta > 1.5 * 3600 else ""))
        if (step + 1) % 5 == 0 or step == 0:
            print(f"第 {st['step']}/{total} 步  奖励 {st['reward_mean']}  采样命中 {st['sample_acc_pct']}%  "
                  f"全相同组 {st['zero_std_frac']}  非法 {st['bad_server_pct'] + st['unparsed_pct']:.1f}%  "
                  f"服务器 1 {st['s1_pct']}%  KL {st['kl']}  {st['elapsed_s'] / 60:.1f} 分钟")
        if (step + 1) % P["eval_every"] == 0 or step + 1 == total:
            run_val(step + 1)
    log_f.close()
    SUMMARY["minutes_train"] = round((time.time() - t0) / 60, 1)
    SUMMARY["val_log"] = VAL_LOG
    with open(os.path.join(OUT, "val_log.json"), "w", encoding="utf-8") as f:
        json.dump(VAL_LOG, f, ensure_ascii=False, indent=1)
    md.save_pretrained(os.path.join(OUT, "lora"))

    if P["final_eval"]:
        # 最终评测：贪心作答写 preds.csv（Mac 上 eval.open_loop 打分），再算完整动作概率
        greedy = greedy_answers(md, EVAL_ROWS)
        write_preds(os.path.join(OUT, "preds.csv"), EVAL_ROWS, greedy)
        with open(os.path.join(OUT, "config.json"), "w", encoding="utf-8") as f:
            json.dump({"name": f"GRPO-{style}（从论文原版 sft）", "model": REF_NAME.format(START) + " + LoRA",
                       "model_tag": "论文原版 1.5B + GRPO", "steps": total, **SUMMARY["settings"]},
                      f, ensure_ascii=False, indent=2)
        lps = action_logprobs(md, DIST_ROWS)
        missing = []
        for r in DIST_ROWS:
            if (r["split"], r["_idx"]) not in greedy:
                missing.append(r)
        if missing:
            greedy.update(greedy_answers(md, missing))
        greedy_dist = {}
        for r in DIST_ROWS:
            key = (r["split"], r["_idx"])
            greedy_dist[key] = greedy[key]
        write_dist(os.path.join(OUT, "dist.jsonl"), DIST_ROWS, lps, greedy_dist)
        s = {}
        for split in ("friend_clean", "test_seen"):
            s[split] = dist_summary(rows_of(DIST_ROWS, split), lps, greedy_dist)
        SUMMARY["dist"] = s
        SUMMARY["final_quick"] = {}
        for split in ("friend_clean", "test_seen"):
            SUMMARY["final_quick"][split] = quick_metrics(rows_of(EVAL_ROWS, split), greedy)
        print(json.dumps(SUMMARY["final_quick"], ensure_ascii=False))
        print(json.dumps(s, ensure_ascii=False, indent=1))

with open(os.path.join(OUT, "summary.json"), "w", encoding="utf-8") as f:
    json.dump(SUMMARY, f, ensure_ascii=False, indent=1)
print("\n全部完成 ✓ 结果目录：", OUT)
print(sorted(os.listdir(OUT)))

In [ ]:
# 第 8 格：冒烟判据（只有 RUN = "smoke" 时有内容）
#   1. "奖励全相同的组"比例：后 10 步的平均要比前 10 步低，或者本来就低于 0.5（说明大多数题都学得到东西）
#   2. 采样回答的平均奖励：后 10 步比前 10 步高
#   3. 验证集贪心准确率：最后一次不低于第 0 步（至少没变差）
if RUN == "smoke":
    def mean_of(logs, key):
        total = 0.0
        for x in logs:
            total += x[key]
        return total / len(logs)

    first = STEP_LOG[:10]
    last = STEP_LOG[-10:]
    z1 = mean_of(first, "zero_std_frac")
    z2 = mean_of(last, "zero_std_frac")
    r1 = mean_of(first, "reward_mean")
    r2 = mean_of(last, "reward_mean")
    a1 = VAL_LOG[0]["acc_pct"]
    a2 = VAL_LOG[-1]["acc_pct"]
    checks = [("全相同组比例 前10步 %.2f → 后10步 %.2f" % (z1, z2), z2 < z1 or z2 < 0.5),
              ("平均奖励 前10步 %.3f → 后10步 %.3f" % (r1, r2), r2 > r1),
              ("验证集准确率 第0步 %.1f%% → 最后 %.1f%%" % (a1, a2), a2 >= a1)]
    print("冒烟判据：")
    for text, ok in checks:
        print(("  ✓ " if ok else "  ✗ ") + text)
    with open(os.path.join(OUT, "smoke_checks.json"), "w", encoding="utf-8") as f:
        out = []
        for text, ok in checks:
            out.append({"check": text, "pass": bool(ok)})
        json.dump(out, f, ensure_ascii=False, indent=1)

## 跑完之后

1. 把第 7 格最后打印的汇总（`diagnose` 是三个模型各一段；训练是最终评测那两段）和第 8 格（`smoke` 时）截图发到对话里。
2. 按运行文档把整个 `<RUN>_<UTC 时间>` 文件夹下载到 Mac 的 `runs/s4/` 里（`lora/` 子文件夹不用下载）。
3. 「代码执行程序 → 断开连接并删除运行时」。